In [11]:
import pandas as pd
import numpy as np
import pulp
import time

In [12]:
df = pd.read_csv('../data/processed/rq3_inputs.csv', index_col='Datetime (UTC)', parse_dates=True)

In [13]:
predicted_prices = df['predicted_price'].values
actual_prices = df['actual_price'].values

predicted_P = df['predicted_P'].values
actual_P = df['actual_P'].values

In [14]:
BATTERY_CAPACITY = 500
MIN_SOC = 0.1 * BATTERY_CAPACITY
MAX_SOC = 0.9 * BATTERY_CAPACITY
MAX_CHARGE_RATE = 100
MAX_DISCHARGE_RATE = 100
CHARGE_EFFICIENCY = 0.95
DISCHARGE_EFFICIENCY = 0.95
HORIZON = 24

In [15]:
def solve_mpc_window(prices, productions, initial_soc, capacity, min_soc, max_soc, max_charge, max_discharge, charge_eff, discharge_eff):

    H = len(prices)
    prob = pulp.LpProblem("battery_mpc", pulp.LpMaximize)

    sell = [pulp.LpVariable(f"sell_{t}", lowBound=0) for t in range(H)]
    charge = [pulp.LpVariable(f"charge_{t}", lowBound=0, upBound=max_charge) for t in range(H)]
    discharge = [pulp.LpVariable(f"discharge_{t}", lowBound=0, upBound=max_discharge) for t in range(H)]
    soc = [pulp.LpVariable(f"soc_{t}", lowBound=min_soc, upBound=max_soc) for t in range(H)]

    prob += pulp.lpSum([prices[t] * (sell[t] + discharge[t] * discharge_eff) / 1000 for t in range(H)])

    for t in range(H):
        prob += sell[t] + charge[t] == productions[t]
        if t == 0:
            prob += soc[t] == initial_soc + charge[t] * charge_eff - discharge[t]
        else:
            prob += soc[t] == soc[t-1] + charge[t] * charge_eff - discharge[t]

    prob.solve(pulp.PULP_CBC_CMD(msg=0))
    return [(sell[t].varValue, charge[t].varValue, discharge[t].varValue) for t in range(H)]

In [16]:
def run_mpc_simulation(predicted_prices, predicted_P, actual_prices, actual_P,
                        capacity, min_soc, max_soc, max_charge, max_discharge,
                        charge_eff, discharge_eff, horizon=24):
    n = len(predicted_prices)
    soc = min_soc
    revenue = 0

    for t in range(n):
        window_end = min(t + horizon, n)
        window_prices = predicted_prices[t:window_end]
        window_prod = predicted_P[t:window_end]

        decisions = solve_mpc_window(
            window_prices, window_prod, soc,
            capacity, min_soc, max_soc,
            max_charge, max_discharge, charge_eff, discharge_eff
        )
        sell_amt, charge_amt, discharge_amt = decisions[0]

        real_price = actual_prices[t]
        real_production = actual_P[t]

        actual_sell = min(sell_amt, real_production)
        actual_charge = min(charge_amt, real_production - actual_sell, max_soc - soc)
        actual_discharge = min(discharge_amt, soc - min_soc)

        surplus = real_production - actual_sell - actual_charge
        actual_sell += surplus

        revenue += real_price * (actual_sell + actual_discharge * discharge_eff) / 1000
        soc = soc + actual_charge * charge_eff - actual_discharge

    return revenue

In [18]:
def perturb_random(values, std_pct, seed=None, min_value=0):

    rng = np.random.default_rng(seed)
    noise = rng.normal(loc=0, scale=std_pct, size=len(values))
    perturbed = values * (1 + noise)
    return np.clip(perturbed, min_value, None)

In [ ]:
N_ITERATIONS = 300

price_uncertainty_std = 0.20
monte_carlo_price_revenues = []

start = time.time()
for i in range(N_ITERATIONS):
    perturbed_prices = perturb_random(predicted_prices, price_uncertainty_std, seed=i)
    revenue = run_mpc_simulation(
        perturbed_prices, predicted_P, actual_prices, actual_P,
        BATTERY_CAPACITY, MIN_SOC, MAX_SOC,
        MAX_CHARGE_RATE, MAX_DISCHARGE_RATE,
        CHARGE_EFFICIENCY, DISCHARGE_EFFICIENCY, HORIZON
    )
    monte_carlo_price_revenues.append(revenue)
    if (i+1) % 50 == 0:
        print(f"{i+1}/{N_ITERATIONS} завршено, поминато време: {time.time()-start:.0f}s")

monte_carlo_price_revenues = np.array(monte_carlo_price_revenues)

In [3]:
print(f"Просечен приход: {monte_carlo_price_revenues.mean():.2f} EUR")
print(f"Стандардна девијација: {monte_carlo_price_revenues.std():.2f} EUR")
print(f"Минимум: {monte_carlo_price_revenues.min():.2f}, Максимум: {monte_carlo_price_revenues.max():.2f}")
print(f"5-ти перцентил: {np.percentile(monte_carlo_price_revenues, 5):.2f}")
print(f"95-ти перцентил: {np.percentile(monte_carlo_price_revenues, 95):.2f}")

Просечен приход: 1131.54 EUR<br>
Стандардна девијација: 15.18 EUR<br>
Минимум: 1012.52<br>
Максимум: 1173.51<br>
5-ти перцентил: 1105.44<br>
95-ти перцентил: 1155.91

In [ ]:
N_ITERATIONS = 300
production_uncertainty_std = 0.20
monte_carlo_production_revenues = []

for i in range(N_ITERATIONS):
    perturbed_P = perturb_random(predicted_P, production_uncertainty_std, seed=i)
    revenue = run_mpc_simulation(
        predicted_prices, perturbed_P, actual_prices, actual_P,
        BATTERY_CAPACITY, MIN_SOC, MAX_SOC,
        MAX_CHARGE_RATE, MAX_DISCHARGE_RATE,
        CHARGE_EFFICIENCY, DISCHARGE_EFFICIENCY, HORIZON
    )
    monte_carlo_production_revenues.append(revenue)

monte_carlo_production_revenues = np.array(monte_carlo_production_revenues)

In [ ]:
print(f"Просечен приход: {monte_carlo_price_revenues.mean():.2f} EUR")
print(f"Стандардна девијација: {monte_carlo_price_revenues.std():.2f} EUR")
print(f"Мин: {monte_carlo_price_revenues.min():.2f}, Макс: {monte_carlo_price_revenues.max():.2f}")
print(f"5-ти перцентил: {np.percentile(monte_carlo_price_revenues, 5):.2f}")
print(f"95-ти перцентил: {np.percentile(monte_carlo_price_revenues, 95):.2f}")

Просечен приход: 1175.55 EUR<br>
Стандардна девијација: 9.80 EUR<br>
Минимум: 1090.09<br>
Максимум: 1173.51<br>
5-ти перцентил: 1159.02 EUR<br>
95-ти перцентил: 1191.28 EUR